In [1]:
import numpy as np
import tensorflow as tf
print(tf.__version__) 
from matplotlib import pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import json
from pathlib import Path

2.21.0


In [ ]:
DATA_DIR = "data_decorr/"

BATCH_SIZE = 32

def build_model(config_size, hidden_nodes, l2):
    initializer = tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0)
    glorot_uniform = tf.keras.initializers.GlorotUniform(seed=42)
    x = tf.keras.Input((config_size,))
    y = tf.keras.layers.Dense(hidden_nodes, activation='sigmoid', kernel_initializer=initializer, kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
    k = tf.keras.layers.Dropout(0.3)(y)
    z = tf.keras.layers.Dense(2, activation='sigmoid')(k)
    model = tf.keras.Model(inputs=x, outputs=z)
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.summary()
    return model

for L in [10, 20, 30, 40, 60]:

    data = np.load(f"{DATA_DIR}/L{L}_ising.npz")
    """split data into input and output"""
    T = data["temperatures"]
    T_c = 2 / np.log(1 + np.sqrt(2))        
    labels = np.transpose(np.array([(T > T_c).astype(int), (T < T_c).astype(int)]))    #create labels from temperature
    configs = data["spins"]

    rng = np.random.default_rng(seed=42)
    idx = np.arange(len(T))
    rng.shuffle(idx)

    # Apply the same permutation to all arrays
    T = T[idx]
    configs = configs[idx]
    labels  = labels[idx, :]
    print(labels.shape)

    """split into training, validation and test data"""
    train_conf, val_conf, test_conf = np.split(configs, [80000, 90000])
    train_label, val_label, test_label = np.split(labels, [80000, 90000], axis=0)
    train_T, val_T, test_T = np.split(T, [80000, 90000])
    #print(train_conf.shape)

    l2 = 1e-4 # / (L ** 2)  # regularization strength λ

    model3_2 = build_model(configs.shape[1], 100, l2)

    w_init, b_init = model3_2.layers[1].get_weights()

    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=8, min_lr=1e-6)
    early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

    history = model3_2.fit(
        train_conf,
        train_label,
        validation_data = (val_conf, val_label),
        batch_size = 128,
        epochs = 150,
        callbacks=[reduce_lr, early_stop]
    )


    file_path = f'models_100_op/training_history_100_op/L{L}.json'
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)

    with open(file_path, 'w') as f:
        json.dump(history.history, f)

    print(f"Successfully saved history to {file_path}")

    # Save after training
    model3_2.save(f"models_100_op/ising_classifier_L{L}.h5")


(100000, 2)


Model: "functional_35"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_35 (InputLayer)     │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_70 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_71 (Dense)                │ (None, 2)              │           202 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,302 (40.24 KB)

 Trainable params: 10,302 (40.24 KB)

...

 Total params: 360,302 (1.37 MB)

 Trainable params: 360,302 (1.37 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 25.4071 - val_loss: 16.3166
Epoch 2/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 10.9339 - val_loss: 6.8502
Epoch 3/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 4.6155 - val_loss: 2.9640
Epoch 4/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2.0605 - val_loss: 1.3093
Epoch 5/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.9079 - val_loss: 0.5357
Epoch 6/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4113 - val_loss: 0.2702
Epoch 7/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2225 - val_loss: 0.1655
Epoch 8/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1390 - val_loss: 0.1125
Epoch 9/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0970 - val_loss: 0.0845
Epoch 10/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0758 - val_loss: 0.0739
Epoch 11/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0599 - val_loss: 0.0657
Epoch 12/80
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/s

Successfully saved history to models_100_op/training_history_100_op/L60.json
